In [33]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [34]:
df=pd.read_csv(r"C:\Users\manda\OneDrive\Desktop\DecisionTree_classifier\RF datasets.csv")


In [35]:
num=df.select_dtypes(include=np.number)
num

,age,salary,experience_years,work_hours_per_week,performance_score,promotion_last_5years,target_left_company
0,56,136748.0,33,47,3.800000,0,1
1,46,25287.0,28,40,2.400000,1,1
2,32,146593.0,3,45,2.970316,0,0
3,60,54387.0,16,47,2.600000,0,0
4,25,28512.0,34,64,1.900000,0,1
...,...,...,...,...,...,...,...
995,22,49241.0,23,35,1.700000,1,1
996,40,84772.0,35,51,4.500000,1,1
997,27,64569.0,5,30,3.900000,1,1
998,61,31745.0,30,33,2.000000,0,1


In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   age                    1000 non-null   int64  
 1   salary                 1000 non-null   float64
 2   experience_years       1000 non-null   int64  
 3   education_level        1000 non-null   object 
 4   department             1000 non-null   object 
 5   city_tier              1000 non-null   object 
 6   work_hours_per_week    1000 non-null   int64  
 7   performance_score      1000 non-null   float64
 8   promotion_last_5years  1000 non-null   int64  
 9   target_left_company    1000 non-null   int64  
dtypes: float64(2), int64(5), object(3)
memory usage: 78.3+ KB


In [37]:
# checking missing valued
df.isnull().sum()

age                      0
salary                   0
experience_years         0
education_level          0
department               0
city_tier                0
work_hours_per_week      0
performance_score        0
promotion_last_5years    0
target_left_company      0
dtype: int64

In [38]:
df.dtypes

age                        int64
salary                   float64
experience_years           int64
education_level           object
department                object
city_tier                 object
work_hours_per_week        int64
performance_score        float64
promotion_last_5years      int64
target_left_company        int64
dtype: object

In [39]:
df.head()

,age,salary,experience_years,education_level,department,city_tier,work_hours_per_week,performance_score,promotion_last_5years,target_left_company
0,56,136748.0,33,PhD,IT,Tier-3,47,3.800000,0,1
1,46,25287.0,28,High School,Finance,Tier-2,40,2.400000,1,1
2,32,146593.0,3,PhD,HR,Tier-1,45,2.970316,0,0
3,60,54387.0,16,High School,IT,Tier-3,47,2.600000,0,0
4,25,28512.0,34,Bachelor,Finance,Tier-2,64,1.900000,0,1


In [40]:
df.describe()

,age,salary,experience_years,work_hours_per_week,performance_score,promotion_last_5years,target_left_company
count,1000.000000,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000
mean,40.986000,85084.980000,19.34500,49.554000,2.970316,0.511000,0.497000
std,13.497852,37146.053131,11.45492,11.255833,1.128795,0.500129,0.500241
min,18.000000,20060.000000,0.00000,30.000000,1.000000,0.000000,0.000000
25%,29.000000,53704.250000,10.00000,40.000000,2.000000,0.000000,0.000000
50%,42.000000,84772.000000,19.00000,49.000000,2.970316,1.000000,0.000000
75%,52.000000,116535.500000,29.00000,59.000000,3.900000,1.000000,1.000000
max,64.000000,149972.000000,39.00000,69.000000,5.000000,1.000000,1.000000


In [41]:
# check the nominal and ordinal columns
df['city_tier'].value_counts()

city_tier
Tier-3    349
Tier-2    336
Tier-1    315
Name: count, dtype: int64

In [42]:
df['department'].value_counts()

department
IT            225
Finance       201
Operations    199
HR            190
Marketing     185
Name: count, dtype: int64

In [43]:
df['education_level'].value_counts()

education_level
High School    299
PhD            244
Master         232
Bachelor       225
Name: count, dtype: int64

In [44]:
from sklearn.preprocessing import OneHotEncoder
ohe=OneHotEncoder(sparse_output=False, drop='first')
cat_cols=['city_tier','department','education_level']
ohe_df=pd.DataFrame(ohe.fit_transform(df[cat_cols]), columns=ohe.get_feature_names_out(cat_cols))
df=pd.concat([df.drop(columns=cat_cols), ohe_df], axis=1)
df.head()


,age,salary,experience_years,work_hours_per_week,performance_score,promotion_last_5years,target_left_company,city_tier_Tier-2,city_tier_Tier-3,department_HR,department_IT,department_Marketing,department_Operations,education_level_High School,education_level_Master,education_level_PhD
0,56,136748.0,33,47,3.800000,0,1,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,46,25287.0,28,40,2.400000,1,1,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,32,146593.0,3,45,2.970316,0,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
3,60,54387.0,16,47,2.600000,0,0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4,25,28512.0,34,64,1.900000,0,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [49]:
from sklearn.model_selection import train_test_split
X=df.drop(columns=['target_left_company'])
y=df['target_left_company']
X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.2, random_state=42)


In [51]:
from sklearn.ensemble import RandomForestClassifier
rf=RandomForestClassifier(n_estimators=100, random_state=42)

rf.fit(X_train, y_train)
y_pred=rf.predict(X_test)
y_pred


array([1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1,
       1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0,
       0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1,
       0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0,
       0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0,
       1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1,
       1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0,
       1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1,
       1, 0])

In [52]:
compare_df=pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
compare_df.head(10)

,Actual,Predicted
521,1,1
737,0,1
740,0,1
660,0,1
411,0,0
678,1,0
626,0,1
513,0,1
859,0,0
136,1,1


In [54]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
accuracy=accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.5
Classification Report:
               precision    recall  f1-score   support

           0       0.46      0.60      0.52        90
           1       0.56      0.42      0.48       110

    accuracy                           0.50       200
   macro avg       0.51      0.51      0.50       200
weighted avg       0.51      0.50      0.50       200

Confusion Matrix:
 [[54 36]
 [64 46]]
